# Pass 1 — Extract Objectives from Every Column

**Goal:** For each fund, send ALL non-empty objective columns in a single API call.  
The LLM extracts objectives **per column independently** — no cross-column reasoning yet.  

**Output:** One row per fund, with per-column extraction results stored as JSON.

In [2]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

INPUT_FILE = Path(config["Input"])
OUTPUT_DIR = Path(config["Output"])

MODEL = "claude-sonnet-4-6"  # UPDATE as needed

OBJECTIVE_COLUMNS = [
    'PRIIPS KID Objective',
    'KIID Objective/Investment Policy',
    'Prospectus Objective',
    'Investment Strategy - English',
    'PRIIPS KID Objective - Danish',
    'PRIIPS KID Objective - Dutch',
    'PRIIPS KID Objective - Finnish',
    'PRIIPS KID Objective - French',
    'PRIIPS KID Objective - German',
    'PRIIPS KID Objective - Italian',
    'PRIIPS KID Objective - Norwegian',
    'PRIIPS KID Objective - Portuguese',
    'PRIIPS KID Objective - Spanish',
    'PRIIPS KID Objective - Swedish',
    'KIID Objective/Investment Policy - German',
    'KIID Objective/Investment Policy - French',
    'KIID Objective/Investment Policy - Italian',
    'KIID Objective/Investment Policy - Spanish',
    'KIID Objective/Investment Policy - Norwegian',
    'KIID Objective/Investment Policy - Swedish',
    'KIID Objective/Investment Policy - Finnish',
    'KIID Objective/Investment Policy - Portuguese',
    'KIID Objective/Investment Policy - Danish',
    'Investment Strategy - Danish',
    'Investment Strategy - Finnish',
    'Investment Strategy - French',
    'Investment Strategy - German',
    'Investment Strategy - Italian',
    'Investment Strategy - Norwegian',
    'Investment Strategy - Portuguese',
    'Investment Strategy - Spanish',
    'Investment Strategy - Swedish',
    'Strategy Description'
]

In [3]:
PASS1_SYSTEM_PROMPT = """You are extracting fund objectives from regulatory disclosure text for European mutual funds.

You will receive MULTIPLE columns of text for a single fund. Extract objectives from EACH column INDEPENDENTLY.
Do NOT cross-reference between columns. Treat each column as a standalone source.

WHAT IS A FUND OBJECTIVE:
The fund objective is the statement of what the fund aims to achieve for its investors — its goal or intended outcome.
Examples: long-term capital growth, regular income, maximizing total returns, beating a benchmark, matching an index.

MULTIPLE OBJECTIVES:
There may be more than one objective per column. Extract ALL and label them separately.
Split objectives when joined by "and", "while", "which also", "that also", or similar.
If one part states a financial goal and another states a sustainability goal, split them.
Examples:
- "provide income and moderate capital growth" → two objectives
- "exceed the performance of the index while maintaining a higher ESG score" → two objectives
- "seek long-term capital growth while reducing the risk of capital loss" → two objectives

Sustainability objectives (extract as separate objectives):
- Reducing greenhouse gas emissions, increasing biodiversity, improving living standards, advancing UN SDGs
- "while maintaining a higher ESG score than the index" or "lower carbon intensity" = separate objective
- Maintaining a minimum share of sustainable investments
- Specific solidarity or social investment commitments (e.g. "invest 5-10% in solidarity enterprises")
- Integration of good governance and sustainable development criteria as a fund-level goal

DO NOT INCLUDE:
- Investment policy/strategy: what the fund invests in, how securities are selected, asset allocation
- Mechanism through which objective is achieved ("by investing in...", "through active management", "through a quality asset strategy")
- Types of companies invested in ("invest in companies that...", "companies whose products...")
- Even if the sentence says "sustainable investment objective", extract only the fund's own intended outcome, not company activities
- Key distinction: A fund-level commitment ("invest 5-10% of assets in solidarity enterprises") IS an objective. A company description ("invest in companies that contribute to the SDGs") is NOT.
- "while taking into account ESG criteria" or "taking into account the risk level" = not an objective
- Risk information, distribution/dividend policy
- Benchmark references used solely for comparison (not as a target to beat)
- Duplicate objectives within the same column

SUSTAINABILITY CONTENT — WHAT TO EXTRACT VS WHAT TO EXCLUDE:

This is the most important judgment call in the extraction. Apply these rules in order:

Step 1 — Is it pure SFDR Article 8/9 boilerplate?
These phrases are required regulatory language and are NEVER an objective on their own:
- "promotes environmental and/or social characteristics"
- "is promoting ESG characteristics"
- "is classified as Article 8 under SFDR"
If the text contains ONLY this boilerplate with no additional specifics, there is no sustainable objective.

Step 2 — Does the text go beyond boilerplate with a specific sustainable commitment?
If boilerplate language is followed by or combined with a specific commitment, extract the commitment as a sustainable objective. The boilerplate framing is excluded; the specific commitment is extracted.

EXTRACT as sustainable objectives:
- "maintains a minimum share of sustainable investments" → sustainable objective
- "solidarity investments of 5-10% in approved solidarity enterprises" → sustainable objective
- "integrating criteria for good governance and sustainable development" → sustainable objective
- "lower carbon intensity than the benchmark index" → sustainable objective
- "reduced greenhouse gas emissions through specific targets" → sustainable objective
- "higher ESG score than the index" → sustainable objective
- "contribute to reducing greenhouse gas emissions" → sustainable objective
- "positive impact on environment and social objectives" → sustainable objective
- Specific sector exclusions framed as a goal (e.g. "exclusion of tobacco, weapons, fossil fuels") → sustainable objective

Do NOT extract as sustainable objectives:
- "taking into account ESG criteria" → approach, not outcome
- "considering sustainability risks" → process, not objective
- "ESG integration in the investment process" → methodology, not objective
- "the fund employs ESG criteria in stock selection" → screening method, not objective
- Generic "promotes environmental and/or social characteristics" without any specifics attached

Step 3 — Does the fund explicitly disclaim sustainable objectives?
If the text states "the fund does not have sustainable investment as its objective" and the sustainability content is framed purely as an approach or consideration (not as a commitment or target), do not extract a sustainable objective.

The key test: Does the text describe something the fund COMMITS TO ACHIEVING (an outcome, a target, a minimum allocation) or something the fund TAKES INTO ACCOUNT (a process, a consideration, a methodology)? Extract the former, exclude the latter.

Worked examples:
- "The fund promotes environmental and social characteristics and maintains a minimum share of sustainable investments under Article 8." → Exclude the boilerplate. EXTRACT "maintains a minimum share of sustainable investments" as sustainable objective.
- "The fund is classified as Article 8 under SFDR and promotes environmental and/or social characteristics." → Pure boilerplate. No sustainable objective.
- "The objective is capital growth, while taking into account ESG criteria." → Only "capital growth" is an objective. "Taking into account ESG criteria" is excluded.
- "The objective is to achieve outperformance while integrating criteria for good governance and sustainable development." → TWO objectives: (1) financial: "achieve outperformance", (2) sustainable: "integrating criteria for good governance and sustainable development"
- "The fund invests 5-10% of its assets in approved solidarity enterprises." → EXTRACT as sustainable objective — fund-level allocation commitment.
- "The fund invests in companies whose products contribute to the SDGs." → Do NOT extract — company description, not fund objective.

TIME HORIZON: If stated, include it (e.g. "over a rolling five-year period").

EXTRACTION RULES:
- Extract text VERBATIM from the source — do not paraphrase
- Classify each objective as "financial" or "sustainable"
- If no objective can be identified in a column, return an empty list for that column
- Detect the language of each column and record it
- If the column is non-English, ALSO provide an English translation of each extracted objective

OUTPUT FORMAT:
Return a JSON object where each key is the exact column name, and the value is:
{
  "language": "English" or "French" or "German" etc.,
  "objectives": [
    {
      "objective_text": "exact verbatim text from source",
      "objective_text_english": "English translation (same as objective_text if already English)",
      "objective_type": "financial" or "sustainable"
    }
  ]
}

If a column has no identifiable objective:
{
  "language": "English",
  "objectives": []
}

IMPORTANT: When extracting verbatim text that contains quotation marks (including German „..." quotes, French «...» quotes, or any other quotation marks), replace them with single quotes in the objective_text field. This is critical to ensure valid JSON output.

"""

In [4]:
PASS1_FEW_SHOT = [
    {
        "fund_name": "Example Multi-Column Fund",
        "columns": {
            "PRIIPS KID Objective": "The Fund aims to maximise the return on your investment through a combination of capital growth and income on the Fund's assets and invest in a manner consistent with the principles of environmental, social and governance (ESG) investing. The Fund invests globally at least 70% of its total assets in the equity securities of companies the main business of which is financial services.",
            "PRIIPS KID Objective - French": "Le Fonds vise à maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus sur les actifs du Fonds et à investir d'une manière conforme aux principes de l'investissement environnemental, social et de gouvernance (ESG). Le Fonds investit à l'échelle mondiale au moins 70 % de son actif total dans les titres de participation de sociétés dont l'activité principale est les services financiers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_text_english": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_type": "financial"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "maximiser le rendement de votre investissement par une combinaison de croissance du capital et de revenus",
                        "objective_text_english": "maximise the return on your investment through a combination of capital growth and income",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Sustainability Split Fund",
        "columns": {
            "PRIIPS KID Objective": "The fund seeks to achieve capital growth and to outperform the benchmark. The fund's sustainable investment objective is to contribute to reducing greenhouse gas emissions. The fund also aims to have long-term positive impact on environment and social objectives."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve capital growth",
                        "objective_text_english": "achieve capital growth",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "outperform the benchmark",
                        "objective_text_english": "outperform the benchmark",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "contribute to reducing greenhouse gas emissions",
                        "objective_text_english": "contribute to reducing greenhouse gas emissions",
                        "objective_type": "sustainable"
                    },
                    {
                        "objective_text": "have long-term positive impact on environment and social objectives",
                        "objective_text_english": "have long-term positive impact on environment and social objectives",
                        "objective_type": "sustainable"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example Norwegian Fund",
        "columns": {
            "PRIIPS KID Objective": "Målsetting\n\nFondets målsetting er å skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK).\n\nFondet skal investere i selskaper globalt som har løsninger på FN's bærekraftsmål og dermed bidrar til omstillingen til et mer bærekraftig samfunn."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "Norwegian",
                "objectives": [
                    {
                        "objective_text": "skape høyest mulig relativ avkastning mot referanseindeksen, MSCI World AC, Net Total Return (målt i NOK)",
                        "objective_text_english": "create the highest possible relative return against the benchmark index, MSCI World AC, Net Total Return (measured in NOK)",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    },
    {
        "fund_name": "Example No-Objective Fund",
        "columns": {
            "PRIIPS KID Objective": "Management objective: Management takes as reference the profitability of the EUROSTOXX 50 Index, solely for informational or comparative purposes. Investment policy: Will invest more than 75% of total exposure in equity assets of European issuers."
        },
        "response": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": []
            }
        }
    },
    {
        "fund_name": "Example Company-Activity Exclusion Fund",
        "columns": {
            "KIID Objective/Investment Policy": "The fund aims to provide capital growth over the long term (5 years or more) by investing in US companies whose products and services are considered by the investment manager as contributing to positive environmental or social change and thereby have an impact on the development of a sustainable global economy."
        },
        "response": {
            "KIID Objective/Investment Policy": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "provide capital growth over the long term (5 years or more)",
                        "objective_text_english": "provide capital growth over the long term (5 years or more)",
                        "objective_type": "financial"
                    }
                ]
            }
        }
    }
]

In [5]:
import re

def robust_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    
    # 1. Strip markdown fences
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    
    # 2. Replace smart quotes with unicode escapes (always, not just after fences)
    cleaned = cleaned.replace('„', '\\u201E')
    cleaned = cleaned.replace('\u201c', '\\u201C')
    cleaned = cleaned.replace('\u201d', '\\u201D')
    cleaned = cleaned.replace('«', '\\u00AB')
    cleaned = cleaned.replace('»', '\\u00BB')
    cleaned = cleaned.replace('‚', '\\u201A')
    cleaned = cleaned.replace('\u2018', '\\u2018')
    cleaned = cleaned.replace('\u2019', '\\u2019')
    
    # 3. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 4. Fix unescaped control characters
    def fix_strings(match):
        s = match.group(0)
        s = s.replace('\n', '\\n')
        s = s.replace('\r', '\\r')
        s = s.replace('\t', '\\t')
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # 5. Last resort — extract outermost { }
    brace_match = re.search(r'\{.*\}', fixed, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass

    return None

In [6]:
def get_nonempty_columns(row, objective_columns):
    """Return dict of only columns that have real content (skip empty/NA)."""
    columns = {}
    for col in objective_columns:
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ['-', 'Not available', '']:
                columns[col] = str(value)
    return columns


def pass1_extract(fund_name, fund_id, columns_dict):
    """Send all non-empty columns for one fund; get per-column extractions back."""
    if not columns_dict:
        return {"_error": "No non-empty columns available"}

    columns_text = "\n\n".join(
        [f"=== Column: {col} ===\n{val}" for col, val in columns_dict.items()]
    )

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

{columns_text}"""

    messages = []
    for ex in PASS1_FEW_SHOT:
        ex_text = "\n\n".join(
            [f"=== Column: {col} ===\n{val}" for col, val in ex["columns"].items()]
        )
        messages.append({"role": "user", "content": f"Fund Name: {ex['fund_name']}\n\n{ex_text}"})
        messages.append({"role": "assistant", "content": json.dumps(ex["response"], indent=2)})

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=8000,
            temperature=0,
            system=PASS1_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        parsed = robust_json_parse(text)
        if parsed is not None:
            return parsed
        return {"_error": f"JSON parse error after all attempts: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [7]:
# === LOAD DATA ===
print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  Total funds: {len(df)}, Columns: {len(df.columns)}")

Loading data...
  Total funds: 5680, Columns: 133


In [8]:
# Adjust sample size as needed: df.sample(n=10, random_state=0) for quick test
#df_sample = df.sample(n=100, random_state=32)
df_sample = df  # Use full dataset

In [9]:
# === RUN PASS 1 ===
pass1_results = []

for idx in tqdm(range(len(df_sample)), desc="Pass 1 — Extract"):
    row = df_sample.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Name']

    columns_dict = get_nonempty_columns(row, OBJECTIVE_COLUMNS)
    result = pass1_extract(fund_name, fund_id, columns_dict)

    pass1_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'columns_sent': list(columns_dict.keys()),
        'num_columns_sent': len(columns_dict),
        'pass1_raw': result  # full per-column JSON
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass1_df = pd.DataFrame(pass1_results)
print(f"\nPass 1 complete: {len(pass1_df)} funds processed")

Pass 1 — Extract:   0%|          | 1/5680 [00:05<8:12:47,  5.21s/it]

   [100% Indice Actions Monde] tokens — in: 3794, out: 311


Pass 1 — Extract:   0%|          | 2/5680 [00:29<25:28:45, 16.15s/it]

   [1895 Aandelen Macro Opportunities FondsD] tokens — in: 10063, out: 1969


Pass 1 — Extract:   0%|          | 3/5680 [00:52<30:28:21, 19.32s/it]

...

   [ACATIS SMALL DIAMONDS X] tokens — in: 3369, out: 244


Pass 1 — Extract:   1%|          | 50/5680 [19:41<16:29:21, 10.54s/it]

   [ACATIS Value und Dividende A] tokens — in: 8969, out: 953


Pass 1 — Extract:   1%|          | 51/5680 [19:44<36:18:39, 23.22s/it]


KeyboardInterrupt: 

In [ ]:
# === FLATTEN FOR INSPECTION ===
# Create a human-readable summary alongside the raw JSON

summary_rows = []
for _, row in pass1_df.iterrows():
    raw = row['pass1_raw']
    if '_error' in raw:
        summary_rows.append({
            'FundId': row['FundId'],
            'Fund_Name': row['Fund_Name'],
            'total_objectives_found': 0,
            'columns_with_objectives': 0,
            'error': raw['_error']
        })
        continue

    total_obj = 0
    cols_with_obj = 0
    for col_name, col_data in raw.items():
        if isinstance(col_data, dict) and 'objectives' in col_data:
            n = len(col_data['objectives'])
            total_obj += n
            if n > 0:
                cols_with_obj += 1

    summary_rows.append({
        'FundId': row['FundId'],
        'Fund_Name': row['Fund_Name'],
        'num_columns_sent': row['num_columns_sent'],
        'total_objectives_found': total_obj,
        'columns_with_objectives': cols_with_obj,
        'error': None
    })

summary_df = pd.DataFrame(summary_rows)
print("PASS 1 SUMMARY:")
print(f"  Funds processed: {len(summary_df)}")
print(f"  Funds with errors: {summary_df['error'].notna().sum()}")
print(f"  Funds with ≥1 objective: {(summary_df['total_objectives_found'] > 0).sum()}")
print(f"  Avg objectives per fund: {summary_df['total_objectives_found'].mean():.1f}")
print(f"  Avg columns with objectives: {summary_df['columns_with_objectives'].mean():.1f}")

PASS 1 SUMMARY:
  Funds processed: 100
  Funds with errors: 0
  Funds with ≥1 objective: 99
  Avg objectives per fund: 10.1
  Avg columns with objectives: 7.3


In [ ]:
# === SAVE PASS 1 OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Save the raw results (pass1_raw as JSON string for portability)
output_df = pass1_df.copy()
output_df['pass1_raw'] = output_df['pass1_raw'].apply(json.dumps)
output_df['columns_sent'] = output_df['columns_sent'].apply(json.dumps)

p1_filename = f'Pass1_Extract_{len(pass1_df)}_funds_{timestamp}.xlsx'
p1_path = os.path.join(OUTPUT_DIR, p1_filename)
output_df.to_excel(p1_path, index=False, engine='openpyxl')
print(f"Saved: {p1_filename}")
print(f"  → Use this file as input to Pass 2")

Saved: Pass1_Extract_100_funds_20260521_1802.xlsx
  → Use this file as input to Pass 2
